In [1]:
import pandas as pd
import gc

# Análise exploratória dos dados de alfabetização do INEP

Os dados de alfabetização baixadados a partir da base do dados do INEP, forma o conjunto principal que irá compor nosso dataset alvo para alimentar o modelo de ML, visto que a variável alvo se encontra neste conjunto de dados, especificamente nos dados dos alunos. Para garantirmos o merge bem sucedido dos diferentes datasets, iremos seguir etapas de qualidade e limpezas dos dados para posteriormente aplicarmos as concatenações e merge de forma eficiente. 

## Ingestão e Verificação dos Dados

Iremos iniciar nossaa análise criando os datasets individuais por área (aluno, município e estado) e ano, para realizarmos as verificações iniciais.

In [15]:
# Dados dos estudantes para cada ano
students2023 = pd.read_csv("alfabetizacao_inep/2023_aluno.csv", sep=";", encoding="latin-1")
students2024 = pd.read_csv("alfabetizacao_inep/2024_aluno.csv", sep=";", encoding="latin-1")
students2025 = pd.read_csv("alfabetizacao_inep/2025_aluno.csv", sep=";", encoding="latin-1")

C:\Users\andre\AppData\Local\Temp\ipykernel_17620\4138985628.py:4: DtypeWarning: Columns (13,14,22,23) have mixed types. Specify dtype option on import or set low_memory=False.
  students2025 = pd.read_csv("alfabetizacao_inep/2025_aluno.csv", sep=";", encoding="latin-1")


In [16]:
# Dados dos municípios por ano
cities2023 = pd.read_csv("alfabetizacao_inep/2023_municipio.csv", sep=";", encoding="latin-1")
cities2024 = pd.read_csv("alfabetizacao_inep/2024_municipio.csv", sep=";", encoding="latin-1")
cities2025 = pd.read_csv("alfabetizacao_inep/2025_municipio.csv", sep=";", encoding="latin-1")

In [17]:
# Dados dos estados por ano
states2023 = pd.read_csv("alfabetizacao_inep/2023_estado.csv", sep=";", encoding="latin-1")
states2024 = pd.read_csv("alfabetizacao_inep/2024_estado.csv", sep=";", encoding="latin-1")
states2025 = pd.read_csv("alfabetizacao_inep/2025_estado.csv", sep=";", encoding="latin-1")

Para entender o DtypeWarning gerado no dataset students2025, iremos entender primeiro de quais colunas se tratam

In [18]:
# Lista os índices apontados no DtypeWarning
indices_erro = [13, 14, 22, 23]
colunas_com_erro = [students2025.columns[i] for i in indices_erro]
colunas_com_erro

['TX_RESPOSTA_BLOCO_1',
 'TX_GABARITO_BLOCO_1',
 'TX_RESPOSTA_BLOCO_4',
 'TX_GABARITO_BLOCO_4']

Por não entendermos que os dados de resposta e gabarito de questões possam atuar como booas variávies de entrada para o modelo, iremos desconsiderar tais colunas a priori. Entendemos que a informação gerada por tais colunas em conjunto já está inserida nas variáveis de saída, não sendo necessário a manutenção de tais dados no dataset que alimentará o modelo

Agora iremos verificar a paridade entre as variáveis de cada conjunto de datasets

In [20]:
print("Todas as colunas em 2023:", students2023.columns.tolist())
print("Todas as colunas em 2024:", students2024.columns.tolist())
print("Todas as colunas em 2025:", students2025.columns.tolist())

Todas as colunas em 2023: ['NU_ANO_AVALIACAO', 'CO_UF', 'SG_UF', 'ID_ALUNO', 'TP_SERIE', 'ID_ESCOLA', 'TP_DEPENDENCIA', 'CO_MUNICIPIO', 'NO_MUNICIPIO', 'IN_PRESENCA_LP', 'IN_PREENCHIMENTO_LP', 'CO_CADERNO_LP', 'VL_PESO_ALUNO_LP', 'VL_PROFICIENCIA_LP', 'IN_ALFABETIZADO']
Todas as colunas em 2024: ['NU_ANO_AVALIACAO', 'CO_UF', 'SG_UF', 'ID_ALUNO', 'TP_SERIE', 'ID_ESCOLA', 'TP_DEPENDENCIA', 'CO_MUNICIPIO', 'NO_MUNICIPIO', 'IN_PRESENCA_LP', 'IN_PREENCHIMENTO_LP', 'CO_CADERNO_LP', 'VL_PESO_ALUNO_LP', 'VL_PROFICIENCIA_LP', 'IN_ALFABETIZADO']
Todas as colunas em 2025: ['NU_ANO_AVALIACAO', 'CO_UF', 'SG_UF', 'ID_ALUNO', 'TP_SERIE', 'ID_ESCOLA', 'TP_DEPENDENCIA', 'CO_MUNICIPIO', 'NO_MUNICIPIO', 'IN_PRESENCA_LP', 'IN_PREENCHIMENTO_LP', 'CO_CADERNO_LP', 'CO_BLOCO_1', 'TX_RESPOSTA_BLOCO_1', 'TX_GABARITO_BLOCO_1', 'CO_BLOCO_2', 'TX_RESPOSTA_BLOCO_2', 'TX_GABARITO_BLOCO_2', 'CO_BLOCO_3', 'TX_RESPOSTA_BLOCO_3', 'TX_GABARITO_BLOCO_3', 'CO_BLOCO_4', 'TX_RESPOSTA_BLOCO_4', 'TX_GABARITO_BLOCO_4', 'VL_PESO

Nos dados referentes aos alunos, notamos que em 2025 há colunas a mais, referentes a

## Consolidação do dataset de alfabetização

Optamos por iniciar nossa abordagem concatenando os arquivos csv de alunos, municípios e estados dos diferentes anos abordados em um único dataset que será analisado e enriquecido com dados complementares

### Dados dos alunos

In [22]:
students2023.describe()

,NU_ANO_AVALIACAO,CO_UF,ID_ALUNO,TP_SERIE,ID_ESCOLA,TP_DEPENDENCIA,CO_MUNICIPIO,IN_PRESENCA_LP,IN_PREENCHIMENTO_LP,CO_CADERNO_LP,VL_PESO_ALUNO_LP,VL_PROFICIENCIA_LP,IN_ALFABETIZADO
count,1747439.0,1.747439e+06,1.747439e+06,1747439.0,1.747439e+06,1.747439e+06,1.747439e+06,1.747439e+06,1.747439e+06,1.747439e+06,1.502809e+06,1.502809e+06,1.747439e+06
mean,2023.0,3.158064e+01,3.164937e+07,2.0,6.002108e+07,2.911219e+00,3.169498e+06,8.601491e-01,8.600066e-01,1.063180e+01,1.160239e+00,7.475638e+02,5.021217e-01
std,0.0,1.080868e+01,1.083407e+07,0.0,1.313293e+04,2.844280e-01,1.082193e+06,3.468324e-01,3.469803e-01,6.185957e+00,4.842851e-01,4.823365e+01,4.999956e-01
min,2023.0,1.100000e+01,1.100000e+07,2.0,6.000000e+07,2.000000e+00,1.100015e+06,0.000000e+00,0.000000e+00,1.000000e+00,9.523810e-02,5.784600e+02,0.000000e+00
25%,2023.0,2.400000e+01,2.400287e+07,2.0,6.000975e+07,3.000000e+00,2.402006e+06,1.000000e+00,1.000000e+00,5.000000e+00,1.000000e+00,7.188371e+02,0.000000e+00
50%,2023.0,3.100000e+01,3.108963e+07,2.0,6.002036e+07,3.000000e+00,3.131307e+06,1.000000e+00,1.000000e+00,1.100000e+01,1.094464e+00,7.521478e+02,1.000000e+00
75%,2023.0,4.100000e+01,4.110591e+07,2.0,6.003424e+07,3.000000e+00,4.119509e+06,1.000000e+00,1.000000e+00,1.600000e+01,1.235216e+00,7.787294e+02,1.000000e+00
max,2023.0,5.200000e+01,5.208382e+07,2.0,6.004281e+07,3.000000e+00,5.222302e+06,1.000000e+00,1.000000e+00,2.100000e+01,1.425454e+02,9.043820e+02,1.000000e+00


In [23]:
students2024.describe()

,NU_ANO_AVALIACAO,CO_UF,ID_ALUNO,TP_SERIE,ID_ESCOLA,TP_DEPENDENCIA,CO_MUNICIPIO,IN_PRESENCA_LP,IN_PREENCHIMENTO_LP,CO_CADERNO_LP,VL_PESO_ALUNO_LP,VL_PROFICIENCIA_LP,IN_ALFABETIZADO
count,2120560.0,2.120560e+06,2.120560e+06,2120560.0,2.120560e+06,2.120560e+06,2.120560e+06,2.120560e+06,2.120560e+06,2.120560e+06,1.851852e+06,1.851852e+06,2.120560e+06
mean,2024.0,3.249435e+01,3.258714e+07,2.0,6.002267e+07,2.867850e+00,3.265448e+06,8.737258e-01,8.732844e-01,1.074422e+01,1.138954e+00,7.490421e+02,5.220880e-01
std,0.0,1.002957e+01,1.003999e+07,0.0,1.216093e+04,3.386892e-01,1.005170e+06,3.321582e-01,3.326542e-01,6.114784e+00,2.056973e-01,4.704344e+01,4.995120e-01
min,2024.0,1.100000e+01,1.100000e+07,2.0,6.000000e+07,2.000000e+00,1.100015e+06,0.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,5.938800e+02,0.000000e+00
25%,2024.0,2.600000e+01,2.604746e+07,2.0,6.001258e+07,3.000000e+00,2.610707e+06,1.000000e+00,1.000000e+00,5.000000e+00,1.030000e+00,7.204800e+02,0.000000e+00
50%,2024.0,3.300000e+01,3.307758e+07,2.0,6.002467e+07,3.000000e+00,3.304524e+06,1.000000e+00,1.000000e+00,1.100000e+01,1.090000e+00,7.539500e+02,1.000000e+00
75%,2024.0,4.100000e+01,4.101274e+07,2.0,6.003207e+07,3.000000e+00,4.103503e+06,1.000000e+00,1.000000e+00,1.600000e+01,1.190000e+00,7.803700e+02,1.000000e+00
max,2024.0,5.300000e+01,5.302770e+07,2.0,6.004250e+07,4.000000e+00,5.300108e+06,1.000000e+00,1.000000e+00,4.300000e+01,2.983000e+01,9.008900e+02,1.000000e+00


In [24]:
students2025.describe()

,NU_ANO_AVALIACAO,CO_UF,ID_ALUNO,TP_SERIE,ID_ESCOLA,TP_DEPENDENCIA,CO_MUNICIPIO,IN_PRESENCA_LP,IN_PREENCHIMENTO_LP,CO_CADERNO_LP,CO_BLOCO_1,CO_BLOCO_2,CO_BLOCO_3,CO_BLOCO_4,VL_PESO_ALUNO_LP,VL_PROFICIENCIA_LP,IN_ALFABETIZADO
count,2222792.0,2.222792e+06,2.222792e+06,2222792.0,2.222164e+06,2.222164e+06,2.222164e+06,2.222792e+06,2.222792e+06,2.222792e+06,1.567060e+06,1.969921e+06,1.969921e+06,402861.000000,1.966605e+06,1.966605e+06,2.222792e+06
mean,2025.0,3.224848e+01,3.238278e+07,2.0,6.002316e+07,2.875741e+00,3.241169e+06,8.862372e-01,8.847454e-01,1.044537e+01,2.582131e+00,7.820045e+00,6.232211e+00,3.999412,1.130269e+00,7.535743e+02,5.862168e-01
std,0.0,1.004266e+01,1.005153e+07,0.0,1.245339e+04,3.298764e-01,1.006450e+06,3.175230e-01,3.193290e-01,6.216472e+00,2.513431e+00,2.820378e+00,2.379316e+00,2.000106,1.429831e-01,4.800590e+01,4.925107e-01
min,2025.0,1.100000e+01,1.100068e+07,2.0,6.000000e+07,2.000000e+00,1.100015e+06,0.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000,1.000000e+00,5.723559e+02,0.000000e+00
25%,2025.0,2.600000e+01,2.616637e+07,2.0,6.001300e+07,3.000000e+00,2.607901e+06,1.000000e+00,1.000000e+00,5.000000e+00,1.000000e+00,5.000000e+00,5.000000e+00,2.000000,1.041667e+00,7.282354e+02,0.000000e+00
50%,2025.0,3.300000e+01,3.304613e+07,2.0,6.002510e+07,3.000000e+00,3.302502e+06,1.000000e+00,1.000000e+00,1.000000e+01,2.000000e+00,8.000000e+00,6.000000e+00,4.000000,1.095238e+00,7.610449e+02,1.000000e+00
75%,2025.0,3.500000e+01,3.544890e+07,2.0,6.003276e+07,3.000000e+00,3.556206e+06,1.000000e+00,1.000000e+00,1.600000e+01,3.000000e+00,1.000000e+01,8.000000e+00,6.000000,1.181818e+00,7.853025e+02,1.000000e+00
max,2025.0,5.300000e+01,5.306777e+07,2.0,6.004364e+07,3.000000e+00,5.300108e+06,1.000000e+00,1.000000e+00,2.100000e+01,1.300000e+01,1.300000e+01,1.000000e+01,7.000000,2.100000e+01,9.040228e+02,1.000000e+00


Ao analisar os intervalos numéricos da variável ID_ALUNO em cada ano, notamos que grande parte dos IDs se repetem. Como naturalmente a maior parte dos alunos irá se modificar de um ano para o outro, concluímos que o mesmo número de ID foi associado a diferentes alunos de um ano para o outro. Ao analisar os valores de ID_ESCOLA nos dados brutos, percebemos que em muitos casos o mesmo ID se refere a escolas em municípios diferentes, de um ano para o outro. Para evitar que essa igualdade dos IDs para alunos e escolas diferentes crie problemas de identificação ao concatenarmos os três datasets, optamos por criar novas máscaras mesclando os IDs com os respectivos anos listados nos csv, antes da concatenação.

In [25]:
# Adicionando o ano ao ID para garantir a unicidade do ID do aluno
students2023['ID_ALUNO'] = students2023['ID_ALUNO'].astype('Int64').astype(str).str.strip() + '_2023'
students2024['ID_ALUNO'] = students2024['ID_ALUNO'].astype('Int64').astype(str).str.strip() + '_2024'
students2025['ID_ALUNO'] = students2025['ID_ALUNO'].astype('Int64').astype(str).str.strip() + '_2025'

# O mesmo processo para o ID da escola
students2023['ID_ESCOLA'] = students2023['ID_ESCOLA'].astype('Int64').astype(str).str.strip() + '_2023'
students2024['ID_ESCOLA'] = students2024['ID_ESCOLA'].astype('Int64').astype(str).str.strip() + '_2024'
students2025['ID_ESCOLA'] = students2025['ID_ESCOLA'].astype('Int64').astype(str).str.strip() + '_2025'

In [ ]:
all_students = pd.concat([students2023, students2024, students2025], ignore_index=True)

# Apaga os datasets não utilizados e limpa a memória
del students2023, students2024, students2025
gc.collect()

students_columns = {
    'NU_ANO_AVALIACAO': 'year',
    'CO_UF': 'state_code',
    'ID_ALUNO': 'student_id',
    'TP_SERIE': 'grade',
    'ID_ESCOLA': 'school_id',
    'TP_DEPENDENCIA': 'education_network_type',
    'CO_MUNICIPIO': 'city_code',
    'IN_PRESENCA_LP': 'student_is_present_exam',
    'IN_PREENCHIMENTO_LP': 'student_took_exam',
    'VL_PESO_ALUNO_LP': 'student_exam_weight',
    'VL_PROFICIENCIA_LP': 'student_exam_score',
    'IN_ALFABETIZADO': 'student_is_literate',
}

all_students = all_students[list(students_columns.keys())].rename(columns=students_columns)

In [27]:
all_students.sort_values(by=['student_id', 'year'], ascending=[True, True], inplace=True)
all_students

,year,state_code,student_id,grade,school_id,education_network_type,city_code,student_is_present_exam,student_exam_weight,student_exam_score,student_is_literate
3851,2023,11,11000001_2023,2,60000062_2023,3.0,1100023.0,1,1.061728,717.663308,0
1748352,2024,11,11000001_2024,2,60000062_2024,3.0,1100023.0,1,1.060000,719.830000,0
3869,2023,11,11000002_2023,2,60000062_2023,3.0,1100023.0,1,1.061728,765.740768,1
1748783,2024,11,11000002_2024,2,60000062_2024,3.0,1100023.0,1,1.060000,773.130000,1
3848,2023,11,11000003_2023,2,60000062_2023,3.0,1100023.0,1,1.061728,790.363649,1
...,...,...,...,...,...,...,...,...,...,...,...
6090786,2025,53,53067759_2025,2,60043566_2025,2.0,5300108.0,1,1.000000,796.073218,1
6090787,2025,53,53067761_2025,2,60043643_2025,2.0,5300108.0,1,1.315789,765.905079,1
6090788,2025,53,53067764_2025,2,60043339_2025,2.0,5300108.0,1,1.052632,720.720275,0
6090789,2025,53,53067768_2025,2,60043358_2025,2.0,5300108.0,0,NaN,NaN,0


O resultado final é um dataset de alunos com os dados de 2023, 2024 e 2025, com os IDs dos alunos e das escolas com garantia de unicidade e em formato string. Optamos por utilizar a boa prática de apagar os datasets não utilizados e limpar a memória para reduzir o risco de sobrecarga da memória RAM, visto que estamos trabalhando com datasets com grande número de linhas.

### Dados municipais

In [ ]:
all_cities = pd.concat([cities2023, cities2024, cities2025], ignore_index=True)

# Apaga os datasets não utilizados e limpa a memória
del cities2023, cities2024, cities2025
gc.collect()

all_cities

,NU_ANO_AVALIACAO,CO_UF,SG_UF,CO_MUNICIPIO,NO_MUNICIPIO,TP_SERIE,ID_TIPO_REDE,PC_ALUNO_ALFABETIZADO,VL_MEDIA_LP,PC_ALUNO_NIVEL_0_LP,PC_ALUNO_NIVEL_1_LP,PC_ALUNO_NIVEL_2_LP,PC_ALUNO_NIVEL_3_LP,PC_ALUNO_NIVEL_4_LP,PC_ALUNO_NIVEL_5_LP,PC_ALUNO_NIVEL_6_LP,PC_ALUNO_NIVEL_7_LP,PC_ALUNO_NIVEL_8_LP
0,2023,11,RO,1100015,Alta Floresta D'Oeste,2,5,64.55,758.3304,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023,11,RO,1100015,Alta Floresta D'Oeste,2,3,64.55,758.3304,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023,11,RO,1100023,Ariquemes,2,3,62.30,757.0999,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2023,11,RO,1100023,Ariquemes,2,5,62.30,757.0999,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2023,11,RO,1100031,Cabixi,2,5,69.10,767.8763,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36406,2025,52,GO,5222203,Vila Boa,2,5,86.22,773.0700,0.00,0.00,0.00,11.99,1.79,26.86,47.38,11.98,0.00
36407,2025,52,GO,5222302,Vila Propício,2,3,87.57,775.3300,0.00,0.00,1.34,5.41,8.29,31.56,34.43,16.29,2.68
36408,2025,52,GO,5222302,Vila Propício,2,5,87.57,775.3300,0.00,0.00,1.34,5.41,8.29,31.56,34.43,16.29,2.68
36409,2025,53,DF,5300108,Brasília,2,2,64.84,747.3454,2.81,4.07,7.25,8.64,20.04,33.62,18.57,4.08,0.91


In [29]:
cities_columns = {
    'NU_ANO_AVALIACAO': 'year',
    'CO_UF': 'state_code',
    'CO_MUNICIPIO': 'city_code',
    'NO_MUNICIPIO': 'city_name',
    'TP_SERIE': 'grade',
    'ID_TIPO_REDE': 'education_network_type',
    'PC_ALUNO_ALFABETIZADO': 'city_percentage_literate_students',
    'VL_MEDIA_LP': 'city_average_exam_score',
}

all_cities = all_cities[list(cities_columns.keys())].rename(columns=cities_columns)

In [30]:
all_cities.head()

,year,state_code,city_code,city_name,grade,education_network_type,city_percentage_literate_students,city_average_exam_score
0,2023,11,1100015,Alta Floresta D'Oeste,2,5,64.55,758.3304
1,2023,11,1100015,Alta Floresta D'Oeste,2,3,64.55,758.3304
2,2023,11,1100023,Ariquemes,2,3,62.30,757.0999
3,2023,11,1100023,Ariquemes,2,5,62.30,757.0999
4,2023,11,1100031,Cabixi,2,5,69.10,767.8763


### Dados estaduais

In [ ]:

all_states = pd.concat([states2023, states2024, states2025], ignore_index=True)

del states2023, states2024, states2025
gc.collect()

0

In [32]:
states_columns = {
    'NU_ANO_AVALIACAO': 'year',
    'CO_UF': 'state_code',
    'SG_UF': 'state_abbreviation',
    'TP_SERIE': 'grade',
    'ID_TIPO_REDE': 'education_network_type',
    'PC_ALUNO_ALFABETIZADO': 'state_percentage_literate_students',
    'VL_MEDIA_LP': 'state_average_exam_score',
}

all_states = all_states[list(states_columns.keys())].rename(columns=states_columns)

In [33]:
all_cities['education_network_type'].value_counts(dropna=False)

education_network_type
3    16396
5    16019
2     3304
0      692
Name: count, dtype: int64

In [34]:
all_states['education_network_type'].value_counts(dropna=False)

education_network_type
5    76
3    75
2    72
0     2
Name: count, dtype: int64

### Merge dos datasets de alunos, municípios e estados

Antes de realizar o merge, iremos verificar os dados das colunas chave, para garantir não ocorrerão erros durante o processo de junção das planilhas. 

In [35]:
all_students['education_network_type'].value_counts(dropna=False)

education_network_type
3.0    5378617
2.0     711521
NaN        628
4.0         25
Name: count, dtype: int64

In [36]:
all_students.isnull().sum()

year                            0
state_code                      0
student_id                      0
grade                           0
school_id                       0
education_network_type        628
city_code                     628
student_is_present_exam         0
student_exam_weight        769525
student_exam_score         769525
student_is_literate             0
dtype: int64

In [37]:
# 1. Filtramos as bases do IBGE/INEP para manter apenas as redes Estadual (2) e Municipal (3)
cities_filtered = all_cities[all_cities['education_network_type'].isin([2, 3])]
states_filtered = all_states[all_states['education_network_type'].isin([2, 3])]

# 2. Realizamos os cruzamentos
literate_exam = (
    all_students
    .merge(cities_filtered,
           how='left',
           on=['year', 'state_code', 'city_code', 'grade', 'education_network_type'])
    .merge(states_filtered,
           how='left',
           on=['year', 'state_code', 'grade', 'education_network_type'])
)

del all_cities, all_states, all_students
gc.collect()

literate_exam.head(10)

,year,state_code,student_id,grade,school_id,education_network_type,city_code,student_is_present_exam,student_exam_weight,student_exam_score,student_is_literate,city_name,city_percentage_literate_students,city_average_exam_score,state_abbreviation,state_percentage_literate_students,state_average_exam_score
0,2023,11,11000001_2023,2,60000062_2023,3.0,1100023.0,1,1.061728,717.663308,0,Ariquemes,62.30,757.0999,RO,65.17,760.1971
1,2024,11,11000001_2024,2,60000062_2024,3.0,1100023.0,1,1.060000,719.830000,0,Ariquemes,65.62,748.1600,RO,62.56,745.3800
2,2023,11,11000002_2023,2,60000062_2023,3.0,1100023.0,1,1.061728,765.740768,1,Ariquemes,62.30,757.0999,RO,65.17,760.1971
3,2024,11,11000002_2024,2,60000062_2024,3.0,1100023.0,1,1.060000,773.130000,1,Ariquemes,65.62,748.1600,RO,62.56,745.3800
4,2023,11,11000003_2023,2,60000062_2023,3.0,1100023.0,1,1.061728,790.363649,1,Ariquemes,62.30,757.0999,RO,65.17,760.1971
5,2024,11,11000003_2024,2,60000062_2024,3.0,1100023.0,1,1.060000,736.930000,0,Ariquemes,65.62,748.1600,RO,62.56,745.3800
6,2023,11,11000004_2023,2,60000062_2023,3.0,1100023.0,1,1.061728,666.922680,0,Ariquemes,62.30,757.0999,RO,65.17,760.1971
7,2024,11,11000004_2024,2,60000062_2024,3.0,1100023.0,1,1.060000,675.400000,0,Ariquemes,65.62,748.1600,RO,62.56,745.3800
8,2023,11,11000005_2023,2,60000062_2023,3.0,1100023.0,1,1.061728,773.748754,1,Ariquemes,62.30,757.0999,RO,65.17,760.1971
9,2024,11,11000005_2024,2,60000062_2024,3.0,1100023.0,1,1.060000,747.620000,1,Ariquemes,65.62,748.1600,RO,62.56,745.3800


In [38]:
literate_exam.tail(40)

,year,state_code,student_id,grade,school_id,education_network_type,city_code,student_is_present_exam,student_exam_weight,student_exam_score,student_is_literate,city_name,city_percentage_literate_students,city_average_exam_score,state_abbreviation,state_percentage_literate_students,state_average_exam_score
6090751,2025,53,53067674_2025,2,60043396_2025,2.0,5300108.0,1,1.285714,784.028347,1,Brasília,64.84,747.3454,DF,64.84,747.3454
6090752,2025,53,53067675_2025,2,60043592_2025,2.0,5300108.0,0,NaN,NaN,0,Brasília,64.84,747.3454,DF,64.84,747.3454
6090753,2025,53,53067678_2025,2,60043632_2025,2.0,5300108.0,1,1.117647,765.114048,1,Brasília,64.84,747.3454,DF,64.84,747.3454
6090754,2025,53,53067679_2025,2,60043335_2025,2.0,5300108.0,0,NaN,NaN,0,Brasília,64.84,747.3454,DF,64.84,747.3454
6090755,2025,53,53067680_2025,2,60043335_2025,2.0,5300108.0,1,1.142857,772.169441,1,Brasília,64.84,747.3454,DF,64.84,747.3454
6090756,2025,53,53067683_2025,2,60043337_2025,2.0,5300108.0,0,NaN,NaN,0,Brasília,64.84,747.3454,DF,64.84,747.3454
6090757,2025,53,53067684_2025,2,60043337_2025,2.0,5300108.0,1,1.076923,761.546223,1,Brasília,64.84,747.3454,DF,64.84,747.3454
6090758,2025,53,53067685_2025,2,60043338_2025,2.0,5300108.0,1,1.080000,740.898820,0,Brasília,64.84,747.3454,DF,64.84,747.3454
6090759,2025,53,53067687_2025,2,60043340_2025,2.0,5300108.0,1,1.062500,752.573966,1,Brasília,64.84,747.3454,DF,64.84,747.3454
6090760,2025,53,53067688_2025,2,60043340_2025,2.0,5300108.0,1,1.066667,742.682333,0,Brasília,64.84,747.3454,DF,64.84,747.3454


In [39]:
literate_exam.set_index(['student_id', 'year'])


,,state_code,grade,school_id,education_network_type,city_code,student_is_present_exam,student_exam_weight,student_exam_score,student_is_literate,city_name,city_percentage_literate_students,city_average_exam_score,state_abbreviation,state_percentage_literate_students,state_average_exam_score
student_id,year,,,,,,,,,,,,,,,
11000001_2023,2023,11,2,60000062_2023,3.0,1100023.0,1,1.061728,717.663308,0,Ariquemes,62.30,757.0999,RO,65.17,760.1971
11000001_2024,2024,11,2,60000062_2024,3.0,1100023.0,1,1.060000,719.830000,0,Ariquemes,65.62,748.1600,RO,62.56,745.3800
11000002_2023,2023,11,2,60000062_2023,3.0,1100023.0,1,1.061728,765.740768,1,Ariquemes,62.30,757.0999,RO,65.17,760.1971
11000002_2024,2024,11,2,60000062_2024,3.0,1100023.0,1,1.060000,773.130000,1,Ariquemes,65.62,748.1600,RO,62.56,745.3800
11000003_2023,2023,11,2,60000062_2023,3.0,1100023.0,1,1.061728,790.363649,1,Ariquemes,62.30,757.0999,RO,65.17,760.1971
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53067759_2025,2025,53,2,60043566_2025,2.0,5300108.0,1,1.000000,796.073218,1,Brasília,64.84,747.3454,DF,64.84,747.3454
53067761_2025,2025,53,2,60043643_2025,2.0,5300108.0,1,1.315789,765.905079,1,Brasília,64.84,747.3454,DF,64.84,747.3454
53067764_2025,2025,53,2,60043339_2025,2.0,5300108.0,1,1.052632,720.720275,0,Brasília,64.84,747.3454,DF,64.84,747.3454


In [40]:
literate_exam.sort_values(by=['student_id', 'year'], ascending=[True, True], inplace=True)

In [41]:
literate_exam.head(10)

,year,state_code,student_id,grade,school_id,education_network_type,city_code,student_is_present_exam,student_exam_weight,student_exam_score,student_is_literate,city_name,city_percentage_literate_students,city_average_exam_score,state_abbreviation,state_percentage_literate_students,state_average_exam_score
0,2023,11,11000001_2023,2,60000062_2023,3.0,1100023.0,1,1.061728,717.663308,0,Ariquemes,62.30,757.0999,RO,65.17,760.1971
1,2024,11,11000001_2024,2,60000062_2024,3.0,1100023.0,1,1.060000,719.830000,0,Ariquemes,65.62,748.1600,RO,62.56,745.3800
2,2023,11,11000002_2023,2,60000062_2023,3.0,1100023.0,1,1.061728,765.740768,1,Ariquemes,62.30,757.0999,RO,65.17,760.1971
3,2024,11,11000002_2024,2,60000062_2024,3.0,1100023.0,1,1.060000,773.130000,1,Ariquemes,65.62,748.1600,RO,62.56,745.3800
4,2023,11,11000003_2023,2,60000062_2023,3.0,1100023.0,1,1.061728,790.363649,1,Ariquemes,62.30,757.0999,RO,65.17,760.1971
5,2024,11,11000003_2024,2,60000062_2024,3.0,1100023.0,1,1.060000,736.930000,0,Ariquemes,65.62,748.1600,RO,62.56,745.3800
6,2023,11,11000004_2023,2,60000062_2023,3.0,1100023.0,1,1.061728,666.922680,0,Ariquemes,62.30,757.0999,RO,65.17,760.1971
7,2024,11,11000004_2024,2,60000062_2024,3.0,1100023.0,1,1.060000,675.400000,0,Ariquemes,65.62,748.1600,RO,62.56,745.3800
8,2023,11,11000005_2023,2,60000062_2023,3.0,1100023.0,1,1.061728,773.748754,1,Ariquemes,62.30,757.0999,RO,65.17,760.1971
9,2024,11,11000005_2024,2,60000062_2024,3.0,1100023.0,1,1.060000,747.620000,1,Ariquemes,65.62,748.1600,RO,62.56,745.3800


In [42]:
literate_exam.isnull().sum()

year                                       0
state_code                                 0
student_id                                 0
grade                                      0
school_id                                  0
education_network_type                   628
city_code                                628
student_is_present_exam                    0
student_exam_weight                   769525
student_exam_score                    769525
student_is_literate                        0
city_name                               4993
city_percentage_literate_students       4993
city_average_exam_score                 4993
state_abbreviation                     28378
state_percentage_literate_students     28378
state_average_exam_score               28378
dtype: int64